## **05_causal_mask: Don't Look Ahead! Implementing the Causal Mask**

We have built a powerful attention mechanism that allows tokens to communicate.  
However, it has a critical flaw for our purpose: **it's a time-traveler**.

### The Problem: Cheating by Looking at the Future

GPT is an **autoregressive** model — it generates text one token at a time.  
When predicting the next word for "A crane ate...", the decision must be based *only* on tokens seen so far: "A", "crane", "ate".  
It **cannot** be allowed to see the actual answer: "fish".

But look at our attention matrix from the last chapter — every token attends to **all** tokens, including future ones. That's cheating.

**Rule:** A token at position `t` must only communicate with tokens at positions `0, 1, ..., t`.

### The Solution: The Causal Mask

We modify the attention scores **before** softmax by setting future positions to **negative infinity** (`-inf`).

Why `-inf`? Because softmax involves an exponential: $e^x / \sum(e^x)$.  
The exponential of negative infinity: $e^{-\infty} \approx 0$.  
This forces attention weights for future tokens to become **zero**.

In [1]:
import torch
import torch.nn.functional as F

# Scaled scores from the end of the last chapter
# Shape (B, T, T) -> (1, 4, 4) for "A crane ate fish"
scaled_scores = torch.tensor([[
    [ 0.0375,  0.2925,  0.1274,  0.1924],
    [ 0.1260,  0.9822,  0.4280,  0.6433],
    [ 0.0437,  0.3405,  0.1484,  0.2228],
    [ 0.0891,  0.6945,  0.3023,  0.4549]
]])
print("Scaled scores shape:", scaled_scores.shape)
print(scaled_scores)

Scaled scores shape: torch.Size([1, 4, 4])
tensor([[[0.0375, 0.2925, 0.1274, 0.1924],
         [0.1260, 0.9822, 0.4280, 0.6433],
         [0.0437, 0.3405, 0.1484, 0.2228],
         [0.0891, 0.6945, 0.3023, 0.4549]]])


### Step 1: Create the Mask

A **lower-triangular matrix** is perfect for this.  
Ones = positions we keep, Zeros = future positions we block.

In [2]:
T = 4
mask = torch.tril(torch.ones(T, T))
print("--- The Mask ---")
print(mask)

--- The Mask ---
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


Look at the rows:
+ Row 0 ("A") can only see column 0 ("A")
+ Row 1 ("crane") can see column 0 ("A") and 1 ("crane")
+ Row 2 ("ate") can see columns 0, 1, 2
+ Row 3 ("fish") can see all columns

The zeros in the upper-right triangle are the "future" connections we must block.

### Step 2: Apply the Mask

We use `masked_fill` to replace all values with `-inf` wherever the mask is `0`.

In [3]:
masked_scores = scaled_scores.masked_fill(mask == 0, float('-inf'))
print("--- Scores After Masking ---")
print(masked_scores)

--- Scores After Masking ---
tensor([[[0.0375,   -inf,   -inf,   -inf],
         [0.1260, 0.9822,   -inf,   -inf],
         [0.0437, 0.3405, 0.1484,   -inf],
         [0.0891, 0.6945, 0.3023, 0.4549]]])


All the scores corresponding to future positions have been replaced with `-inf`.

### Step 3: Re-run Softmax

Now the magic happens: $e^{-\infty} = 0$, so future tokens get **zero** attention weight.

In [4]:
attention_weights = F.softmax(masked_scores, dim=-1)
print("--- Final Causal Attention Weights ---")
print(attention_weights.data.round(decimals=2))

--- Final Causal Attention Weights ---
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.3000, 0.7000, 0.0000, 0.0000],
         [0.2900, 0.3900, 0.3200, 0.0000],
         [0.1800, 0.3300, 0.2200, 0.2600]]])


This is the "Aha!" moment. The upper-right triangle is now all zeros:
+ "A" can only attend to itself (100%)
+ "crane" attends to "A" (30%) and "crane" (70%)
+ "ate" attends to "A", "crane", and "ate"

Information can now only flow from the past to the present.

| Attention Type | "crane" attends to "fish"? | "ate" attends to "fish"? |
| :--- | :--- | :--- |
| Unmasked (Ch 5) | Yes | Yes |
| **Causal (Ch 6)** | **No (0%)** | **No (0%)** |

### Encapsulating in the `nn.Module`

**`register_buffer`** — stores the mask as part of the model's state.  
A **buffer** is a tensor that:
+ Gets saved with the model state
+ Moves to the GPU with `.to(device)`  
+ Is **NOT** updated by the optimizer (it's fixed, not trainable)

Perfect for our fixed causal mask.

In [5]:
import torch.nn as nn
import math
from dataclasses import dataclass

@dataclass
class GPTConfig:
    n_embd: int = 2
    block_size: int = 8

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)

        # Register the mask as a buffer (not a parameter)
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        scaled_scores = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))

        # THE CAUSAL MASK: slice the buffer to match current sequence length
        scaled_scores = scaled_scores.masked_fill(
            self.bias[:, :, :T, :T] == 0, float("-inf")
        )

        attention_weights = F.softmax(scaled_scores, dim=-1)
        output = attention_weights @ v
        return output

# Test it
config = GPTConfig(n_embd=2, block_size=8)
model = CausalSelfAttention(config)

x = torch.tensor([
    [[0.1, 0.1], [1.0, 0.2], [0.1, 0.9], [0.8, 0.0]]
]).float()

output = model(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output:\n", output)

Input shape: torch.Size([1, 4, 2])
Output shape: torch.Size([1, 1, 4, 2])
Output:
 tensor([[[[-0.1383, -0.1130],
          [-0.4756, -0.4314],
          [-0.5444, -0.4489],
          [-0.5531, -0.4706]]]], grad_fn=<UnsafeViewBackward0>)


We have now built a fully functional **causal**, single-headed attention mechanism.  
It can learn to find context, but it can **no longer cheat** by looking into the future.

But it's still just a single mechanism — like one person in a meeting trying to track everything at once: grammar, meaning, long-range context...  
To make it truly powerful, we need **many conversations at the same time**: **Multi-Head Attention**.